In [10]:
import pandas as pd

import sys
import os

# Make config.py (in the repo root, one level up) importable
sys.path.append(os.path.abspath(".."))

import importlib
import config
importlib.reload(config)

<module 'config' from '/Users/qian/KWF/rainforest-audio-detection/config.py'>

In [11]:
labels = pd.read_csv(config.LABELS_PROGRESS_PATH)

In [2]:
fullpred_path = "/Users/qian/KWF/Team Jacama/Stage1_py/evaluation_results/full_predictions.csv"

fp_df = pd.read_csv(fullpred_path)
print("Shape:", fp_df.shape)
print()
print("Columns:", list(fp_df.columns))
print()
print(fp_df[["clip_name", "model_confidence"]].head())

Shape: (631317, 8)

Columns: ['clip_name', 'Recorder', 'confidence', 'Human Activity Score', 'audio_path', 'ground_truth', 'prediction', 'model_confidence']

                          clip_name  model_confidence
0  Audio_Moth_1_20250317_093112.wav          0.170993
1  Audio_Moth_1_20250317_093115.wav          0.004758
2  Audio_Moth_1_20250317_093118.wav          0.472877
3  Audio_Moth_1_20250317_093121.wav          0.279459
4  Audio_Moth_1_20250317_093124.wav          0.999982


In [4]:
# Join model scores onto labels by clip_name
score_lookup = fp_df[["clip_name", "model_confidence"]]
labels = labels.merge(score_lookup, on="clip_name", how="left")

# Check the join worked — how many labels rows got a score?
print("Total labels rows:", len(labels))
print("Rows with a model score:", labels["model_confidence"].notna().sum())
print("Rows missing a model score:", labels["model_confidence"].isna().sum())

Total labels rows: 631317
Rows with a model score: 631317
Rows missing a model score: 0


In [5]:
# Look at model scores for ONLY the unknown pool
unknown_pool = labels[labels["meaningful"] == "unknown"]
print("Unknown pool size:", len(unknown_pool))
print()
print("Model score distribution in the unknown pool:")
print(unknown_pool["model_confidence"].describe())
print()

# How many fall into low-score (background candidate) ranges?
for thresh in [0.01, 0.05, 0.1, 0.2]:
    n = (unknown_pool["model_confidence"] < thresh).sum()
    print(f"Unknown clips with model score < {thresh}: {n:,} ({n/len(unknown_pool)*100:.1f}%)")

Unknown pool size: 620446

Model score distribution in the unknown pool:
count    620446.000000
mean          0.626553
std           0.339226
min           0.000000
25%           0.398539
50%           0.638481
75%           0.976843
max           1.000000
Name: model_confidence, dtype: float64

Unknown clips with model score < 0.01: 33,592 (5.4%)
Unknown clips with model score < 0.05: 60,243 (9.7%)
Unknown clips with model score < 0.1: 78,717 (12.7%)
Unknown clips with model score < 0.2: 106,179 (17.1%)


In [7]:
# Low-score background audit sampler
# Sample unknown clips from low model-score sub-bands, copy audio for listening.
import os
import shutil
import numpy as np

# --- Config ---
N_PER_BAND = 20
RANDOM_SEED = 42

# Low-score sub-bands to sample from
score_bands = [
    (0.00, 0.01, "0.00-0.01"),
    (0.01, 0.03, "0.01-0.03"),
    (0.03, 0.05, "0.03-0.05"),
    (0.05, 0.10, "0.05-0.10"),
]

OUT_DIR = os.path.join(config.PROJECT_ROOT, "outputs", "lowscore_audit")
AUDIO_OUT = os.path.join(OUT_DIR, "audio")
os.makedirs(AUDIO_OUT, exist_ok=True)

# --- Sample from each band (within the unknown pool) ---
rng = np.random.RandomState(RANDOM_SEED)
unknown_pool = labels[labels["meaningful"] == "unknown"]

sampled = []
for low, high, name in score_bands:
    cell = unknown_pool[
        (unknown_pool["model_confidence"] >= low) &
        (unknown_pool["model_confidence"] < high)
    ].copy()
    cell["score_band"] = name
    n_take = min(N_PER_BAND, len(cell))
    sampled.append(cell.sample(n=n_take, random_state=rng))

sample_df = pd.concat(sampled, ignore_index=True)
print("Total sampled:", len(sample_df))
print(sample_df["score_band"].value_counts().sort_index())

# --- Copy audio, build review rows ---
review_rows = []
missing = 0
for _, row in sample_df.iterrows():
    src = row["audio_path"]
    clip_name = row["clip_name"]
    out_name = "{}_{}".format(row["score_band"], clip_name)

    if os.path.exists(src):
        shutil.copy2(src, os.path.join(AUDIO_OUT, out_name))
    else:
        missing += 1
        continue

    review_rows.append({
        "clip_name": clip_name,
        "recorder": row["Recorder"],
        "score_band": row["score_band"],
        "model_confidence": row["model_confidence"],
        "audio_file": "audio/" + out_name,
        "category": "",   # fill while listening: meaningful / background / unsure
        "notes": "",
    })

review_df = pd.DataFrame(review_rows)
review_df = review_df.sort_values(["score_band", "model_confidence"])

csv_path = os.path.join(OUT_DIR, "lowscore_audit.csv")
review_df.to_csv(csv_path, index=False)

print("\nDone.")
print("  Audio copied to:", AUDIO_OUT)
print("  Review CSV:     ", csv_path)
if missing:
    print(f"  [WARN] {missing} audio files missing from source")

Total sampled: 80
score_band
0.00-0.01    20
0.01-0.03    20
0.03-0.05    20
0.05-0.10    20
Name: count, dtype: int64

Done.
  Audio copied to: /Users/qian/KWF/rainforest-audio-detection/outputs/lowscore_audit/audio
  Review CSV:      /Users/qian/KWF/rainforest-audio-detection/outputs/lowscore_audit/lowscore_audit.csv


In [8]:
import os
import pandas as pd
from IPython.display import display, Audio, clear_output
import ipywidgets as widgets

# Paths
AUDIT_DIR = os.path.join(config.PROJECT_ROOT, "outputs", "lowscore_audit")
AUDIT_CSV = os.path.join(AUDIT_DIR, "lowscore_audit.csv")

# Load
adf = pd.read_csv(AUDIT_CSV)
adf["category"] = adf["category"].fillna("").astype(str)
adf["notes"] = adf["notes"].fillna("").astype(str)

print(f"Total clips to review: {len(adf)}")
print(f"Already tagged: {(adf['category'] != '').sum()}")
print(f"Remaining: {(adf['category'] == '').sum()}")

current_idx = [0]
CATEGORIES = ["meaningful", "background", "unsure"]

category_dropdown = widgets.Dropdown(options=[""] + CATEGORIES, description="Category:", value="")
notes_text = widgets.Text(description="Notes:", placeholder="optional")
prev_btn = widgets.Button(description="◀ Prev")
next_btn = widgets.Button(description="Next ▶", button_style="primary")
save_btn = widgets.Button(description="💾 Save CSV", button_style="success")
jump_input = widgets.IntText(value=0, description="Jump to:")
jump_btn = widgets.Button(description="Go")
status_label = widgets.Label(value="")
output = widgets.Output()

def show_clip(idx):
    with output:
        clear_output(wait=True)
        if idx < 0 or idx >= len(adf):
            print("Out of range.")
            return
        row = adf.iloc[idx]
        n_tagged = (adf["category"] != "").sum()
        print(f"Clip {idx + 1} / {len(adf)}  |  Tagged so far: {n_tagged} / {len(adf)}")
        print(f"File: {row['clip_name']}")
        print(f"Recorder: {row['recorder']}  |  Score band: {row['score_band']}  |  Model score: {row['model_confidence']:.4f}")
        print(f"Current tag: '{row['category']}'  |  Notes: '{row['notes']}'")
        print()
        audio_path = os.path.join(AUDIT_DIR, row["audio_file"])
        if os.path.exists(audio_path):
            display(Audio(filename=audio_path))
        else:
            print(f"[Audio not found: {audio_path}]")
    category_dropdown.value = row["category"] if row["category"] in [""] + CATEGORIES else ""
    notes_text.value = row["notes"]

def save_current_tags():
    adf.at[current_idx[0], "category"] = category_dropdown.value
    adf.at[current_idx[0], "notes"] = notes_text.value

def on_next(b):
    save_current_tags()
    if current_idx[0] < len(adf) - 1:
        current_idx[0] += 1
        show_clip(current_idx[0])
    else:
        status_label.value = "Reached the end. Don't forget to save."

def on_prev(b):
    save_current_tags()
    if current_idx[0] > 0:
        current_idx[0] -= 1
        show_clip(current_idx[0])

def on_save(b):
    save_current_tags()
    adf.to_csv(AUDIT_CSV, index=False)
    n_tagged = (adf["category"] != "").sum()
    status_label.value = f"✓ Saved. {n_tagged}/{len(adf)} tagged."

def on_jump(b):
    save_current_tags()
    target = jump_input.value
    if 0 <= target < len(adf):
        current_idx[0] = target
        show_clip(current_idx[0])

next_btn.on_click(on_next)
prev_btn.on_click(on_prev)
save_btn.on_click(on_save)
jump_btn.on_click(on_jump)

controls = widgets.HBox([prev_btn, next_btn, save_btn])
jump_row = widgets.HBox([jump_input, jump_btn])
ui = widgets.VBox([output, category_dropdown, notes_text, controls, jump_row, status_label])

display(ui)
show_clip(current_idx[0])

Total clips to review: 80
Already tagged: 0
Remaining: 80


In [9]:
# Aggregate the low-score audit results
adf = pd.read_csv(AUDIT_CSV)  # reload to be safe
adf["category"] = adf["category"].fillna("").str.strip().str.lower()

print("Overall category distribution:")
print(adf["category"].value_counts())
print()

print("Category by score band:")
print(pd.crosstab(adf["score_band"], adf["category"]))

Overall category distribution:
category
    80
Name: count, dtype: int64

Category by score band:
category      
score_band    
0.00-0.01   20
0.01-0.03   20
0.03-0.05   20
0.05-0.10   20
